<a href="https://colab.research.google.com/github/DrKroll/llm-humanidades-salamanca/blob/main/notebooks/taller_llm/01_preparar_y_explorar_tus_textos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 01 · Preparar el entorno y explorar tus textos

⏱️ **≈ 30 min** (martes 13:00–14:00, después de presentar el repositorio)

| Parte | Contenido | Tiempo |
|---|---|---|
| A | Preparación: GPU, configuración, prueba rápida | ≈ 10 min |
| B | Explorar tus textos: estadísticas, palabras frecuentes, KWIC; subir tu propio corpus | ≈ 20 min |

# Parte A · Preparación del entorno

**Objetivo:** comprobar que todo funciona antes de empezar a entrenar modelos.

En este taller (4 cuadernos) vas a:
1. **01** · preparar el entorno y explorar tus propios textos,
2. **02** · crear un *tokenizador*, entrenar un GPT diminuto **desde cero** y jugar con la generación (temperatura, top-k, top-p),
3. **03** · ajustar (*fine-tuning*) un modelo pequeño ya existente con tus textos,
4. **04** · hacer preguntas sobre el teatro de Calderón: modelo base, RAG y ajuste fino.

Para quien vaya rápido o quiera seguir en casa: carpeta `notebooks/extras/`.

### Antes de empezar en Google Colab
1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4** (gratuita).
2. Ejecuta las celdas en orden con el botón ▶ o con **Mayús + Intro**.
3. Si algo falla: **Entorno de ejecución → Reiniciar sesión** y vuelve a ejecutar desde arriba.

> 💡 No hace falta entender todo el código. Las celdas llaman a funciones del paquete `dhllm`
> (carpeta `src/dhllm/`). Lo importante son los **ajustes** y los **resultados**.

In [ ]:
#@title ⚙️ Preparación: ejecuta esta celda primero (no hace falta leer el código) { display-mode: "form" }
# ⚙️ Preparación: ejecuta esta celda primero (Mayús + Intro). Puedes repetirla sin problema.
import os, sys, importlib.util
REPO_URL = "https://github.com/DrKroll/llm-humanidades-salamanca"
REPO_DIR = "/content/llm-humanidades-salamanca"

if "google.colab" in sys.modules:
    # En Colab: descargamos el repositorio (solo la primera vez) y entramos en su carpeta
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    # Colab ya trae torch, transformers, peft, etc.: solo instalamos lo que falte,
    # sin tocar las versiones preinstaladas
    paquetes = {"transformers": "transformers", "tokenizers": "tokenizers", "peft": "peft",
                "accelerate": "accelerate", "yaml": "pyyaml", "networkx": "networkx", "tqdm": "tqdm"}
    faltan = [pip for mod, pip in paquetes.items() if importlib.util.find_spec(mod) is None]
    if faltan:
        !pip install -q {" ".join(faltan)}
else:
    # En tu ordenador: subimos de carpeta hasta encontrar config.yaml
    while not os.path.exists("config.yaml") and os.getcwd() != "/":
        os.chdir("..")
# El código del taller (carpeta src/dhllm) se importa directamente, sin reiniciar
if os.path.abspath("src") not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))
print("Carpeta de trabajo:", os.getcwd())

## 1. ¿Qué ordenador tenemos?
La siguiente celda muestra las versiones instaladas y si hay una GPU disponible.

In [ ]:
from dhllm import describe_environment
info = describe_environment()

## 2. La configuración

Todos los ajustes del taller están en el archivo **`config.yaml`** (en Colab: icono de carpeta 📁 a la izquierda).
Además, cada cuaderno tiene arriba una celda de **Ajustes** donde puedes cambiar valores sin tocar ningún archivo.

In [ ]:
from dhllm import load_config
cfg = load_config()
print("Carpetas de textos:", cfg.data.folders)
print("Tamaño del vocabulario:", cfg.tokenizer.vocab_size)
print("Modelo base para el ajuste fino:", cfg.finetune.base_model)

## 3. Una prueba rápida
Si esta celda muestra un número y un mensaje, ¡todo está listo!

In [ ]:
import torch
x = torch.rand(3, 3)
print("Suma de una matriz aleatoria:", round(x.sum().item(), 3))
print("✅ Todo listo. Continúa con la parte B.")

### ✅ Comprobación
Si estás en Colab sin GPU, cambia el tipo de entorno a **T4** y vuelve a ejecutar la celda del punto 1.

# Parte B · Explora tus textos

**Objetivo:** cargar un corpus (colección de textos) y conocerlo un poco antes de entrenar.

Un modelo de lenguaje solo puede aprender lo que hay en sus datos. Por eso, en Humanidades Digitales
lo primero es siempre **mirar el corpus**: ¿cuánto texto hay?, ¿qué palabras aparecen?, ¿hay «ruido»
(números de página, cabeceras, errores de OCR)?

Usaremos como ejemplo la *Primera parte del Quijote* y el *Lazarillo de Tormes* (dominio público,
Project Gutenberg). Puedes usar tus propios textos `.txt`.

## ⚙️ Ajustes
Cambia aquí la carpeta con tus textos. `data/sample` contiene los textos de ejemplo.

In [ ]:
CARPETAS = ["data/sample"]     # p. ej. ["data/mis_textos"]
PALABRA_CLAVE = "Dulcinea"     # para buscar en contexto (KWIC)

### (Opcional) Sube tus propios textos
En Colab, ejecuta la celda siguiente con `SUBIR = True` para elegir archivos `.txt` de tu ordenador.
Se guardarán en `data/mis_textos/`. Después cambia `CARPETAS = ["data/mis_textos"]` arriba.

In [ ]:
SUBIR = False
if SUBIR:
    from dhllm.data import upload_texts
    upload_texts("data/mis_textos")

## 1. Cargar los textos

In [ ]:
from dhllm.data import load_texts
textos = load_texts(CARPETAS)
textos

Veamos el principio del primer texto:

In [ ]:
print(textos[0].content[:800])

## 2. Estadísticas básicas
- **riqueza léxica** = palabras distintas ÷ palabras totales (*type-token ratio*). Depende mucho de la longitud del texto.

In [ ]:
from dhllm.data import corpus_stats
tabla = corpus_stats(textos)

## 3. Palabras más frecuentes
Quitamos las *palabras vacías* (de, la, que, y...) para ver las palabras con contenido.

In [ ]:
from dhllm.data import top_words
top_words(textos, n=25);

## 4. Palabra clave en contexto (KWIC)
Una herramienta clásica de la lingüística de corpus.

In [ ]:
from dhllm.data import kwic
kwic(textos, PALABRA_CLAVE, width=45, max_results=8);

## 5. ¿Cuánto texto necesita un modelo?
Para comparar: GPT-3 se entrenó con unos 300 000 millones de tokens; los modelos actuales con más de
10 billones. Nuestro corpus de ejemplo tiene ~1 millón de caracteres. Por eso nuestros modelos serán
**miniaturas**: aprenderán algo del estilo, pero no «sabrán» cosas.

In [ ]:
from dhllm.data import join_texts
corpus = join_texts(textos)
print(f"Caracteres: {len(corpus):,}".replace(",", "."))
print(f"Palabras aprox.: {len(corpus.split()):,}".replace(",", "."))

## 🖐️ Tu turno
1. Busca otras palabras con `kwic` (por ejemplo *molinos*, *honra*, *pan*). ¿Qué te dice el contexto?
2. Cambia `n=25` por `n=50` en `top_words`. ¿Aparecen nombres propios?
3. Sube un texto propio y compara sus estadísticas con las del Quijote.
4. **Para pensar:** ¿qué «ruido» ves en tus textos que un modelo aprendería a imitar?